In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
!pip install -q transformers safetensors loralib soundfile

import os
if not os.path.exists('./vox-profile-release/src'):
    !rm -rf ./vox-profile-release
    !git clone https://github.com/tiantiaf0627/vox-profile-release.git ./vox-profile-release
    print("Cloned vox-profile-release from GitHub")
else:
    print("vox-profile-release/src already exists")

from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="tiantiaf/whisper-large-v3-voice-quality",
    local_dir="./vox-profile-release",
    local_dir_use_symlinks=False
)
print("Model weights downloaded.")


In [ ]:
import os, sys, torch, torch.nn as nn
import soundfile as sf


sys.path.insert(0, os.path.abspath('./vox-profile-release'))


from src.model.voice_quality import whisper_voice_quality as _vq_mod
_orig_layer_fwd = _vq_mod.WhisperEncoderLayer.forward

def _patched_layer_fwd(self, hidden_states, attention_mask=None, layer_head_mask=None, output_attentions=False, **kwargs):
    return _orig_layer_fwd(self, hidden_states, attention_mask, layer_head_mask, output_attentions)

_vq_mod.WhisperEncoderLayer.forward = _patched_layer_fwd
print("Patched WhisperEncoderLayer (layer_head_mask default=None)")

from src.model.voice_quality.whisper_voice_quality import WhisperWrapper

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading model on {device}...")
model = WhisperWrapper.from_pretrained("tiantiaf/whisper-large-v3-voice-quality").float().to(device)
model.eval()
print("Model loaded.")

VOICE_QUALITY_LABELS = [
    'shrill', 'nasal', 'deep',
    'silky', 'husky', 'raspy', 'guttural', 'vocal-fry',
    'booming', 'authoritative', 'loud', 'hushed', 'soft',
    'crisp', 'slurred', 'lisp', 'stammering',
    'singsong', 'pitchy', 'flowing', 'monotone', 'staccato', 'punctuated', 'enunciated', 'hesitant',
]

SR = 16000
MAX_LEN = 15 * SR

def load_audio(path):
    data, sr = sf.read(path, dtype='float32')
    wav = torch.from_numpy(data)
    if wav.dim() > 1:
        wav = wav.mean(dim=-1)
    if sr != SR:
        import torchaudio
        wav = torchaudio.functional.resample(wav, sr, SR)
    if wav.shape[0] < 3 * SR:
        return None
    return wav[:MAX_LEN]

def predict(audio, threshold=0.5):
    data = audio.unsqueeze(0).float().to(device)
    with torch.no_grad():
        logits = model(data, return_feature=False)
    probs = nn.Sigmoid()(torch.tensor(logits)).cpu().numpy()[0]
    return [VOICE_QUALITY_LABELS[i] for i, p in enumerate(probs) if p > threshold]

print("Predict function ready.")


In [ ]:
import os, json, time
from tqdm import tqdm

DRIVE_ROOT = '/content/drive/MyDrive/librimix_3000'
FRESH_START = True

CONDITIONS = {
    'clean_gt':              os.path.join(DRIVE_ROOT, 's1'),
    'mixture':               os.path.join(DRIVE_ROOT, 'mix_both'),
    'noisy_40':              os.path.join(DRIVE_ROOT, 's1_noisy_40'),
    'reuse_mix':             os.path.join(DRIVE_ROOT, 'after_reuse_mix'),
    'reuse_s1':              os.path.join(DRIVE_ROOT, 'after_reuse_s1'),
    'reuse_s1_noisy_40':     os.path.join(DRIVE_ROOT, 'after_reuse_s1_noisy_40'),
}

RESULTS_PATH = os.path.join(DRIVE_ROOT, 'voice_quality_results_3000.json')

if FRESH_START:
    results = {}
    if os.path.exists(RESULTS_PATH):
        os.remove(RESULTS_PATH)
        print("Deleted old results file (FRESH_START=True)")
elif os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f"Resuming from existing results.")
else:
    results = {}


for cond_name, cond_dir in CONDITIONS.items():
    if not os.path.exists(cond_dir):
        print(f"\n  {cond_name}: NOT AVAILABLE YET (RE-USE still running?) — skipping")
        continue

    wav_files = sorted([f for f in os.listdir(cond_dir) if f.endswith('.wav')])

    if cond_name not in results:
        results[cond_name] = {}

    already_done = set(results[cond_name].keys())
    remaining = [f for f in wav_files if f not in already_done]

    print(f"\n{'='*60}")
    print(f"  {cond_name} ({cond_dir.split('/')[-1]})")
    print(f"  Total: {len(wav_files)} | Done: {len(already_done)} | Remaining: {len(remaining)}")
    print(f"{'='*60}")

    if not remaining:
        print("  All done — skipping.")
        continue

    errors = 0
    start_time = time.time()

    for i, fname in enumerate(tqdm(remaining, desc=cond_name)):
        try:
            audio = load_audio(os.path.join(cond_dir, fname))
            if audio is None:
                results[cond_name][fname] = []
            else:
                results[cond_name][fname] = predict(audio)
        except Exception as e:
            results[cond_name][fname] = []
            errors += 1
            if errors <= 3:
                print(f"\n  ERROR {fname}: {e}")

        if (i + 1) % 200 == 0:
            with open(RESULTS_PATH, 'w') as f:
                json.dump(results, f)
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            eta = (len(remaining) - i - 1) / rate / 60
            print(f"  [{i+1}/{len(remaining)}] {rate:.1f} files/sec, ETA: {eta:.0f} min — saved checkpoint")

    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f)

    elapsed = time.time() - start_time
    print(f"  Done: {len(remaining) - errors} files in {elapsed/60:.1f} min ({errors} errors)")

print(f"\n{'='*60}")
print(f"Results saved to: {RESULTS_PATH}")
for c in CONDITIONS:
    n = len(results.get(c, {}))
    print(f"  {c}: {n} files")
print(f"{'='*60}")


In [ ]:
print("Re-run Cell 4 above to pick up any newly available RE-USE outputs.")
print("It will skip conditions that are already fully processed.")


In [ ]:
import os, json
from collections import Counter

DRIVE_ROOT = '/content/drive/MyDrive/librimix_3000'
RESULTS_PATH = os.path.join(DRIVE_ROOT, 'voice_quality_results_3000.json')

with open(RESULTS_PATH) as f:
    results = json.load(f)

VOICE_QUALITY_LABELS = [
    'shrill', 'nasal', 'deep',
    'silky', 'husky', 'raspy', 'guttural', 'vocal-fry',
    'booming', 'authoritative', 'loud', 'hushed', 'soft',
    'crisp', 'slurred', 'lisp', 'stammering',
    'singsong', 'pitchy', 'flowing', 'monotone', 'staccato', 'punctuated', 'enunciated', 'hesitant',
]

print(f"{'='*80}")
print(f"  VOICE QUALITY EVALUATION — 3,000 LibriMix Samples")
print(f"{'='*80}")


for cond in results:
    n = len(results[cond])
    counts = Counter()
    for fname, labels in results[cond].items():
        counts.update(labels)
    print(f"\n  {cond} ({n} samples):")
    for lbl, cnt in counts.most_common(10):
        print(f"    {lbl:<15} {cnt:>5} ({cnt/n*100:.1f}%)")


gt = results.get('clean_gt', {})
comparisons = [
    ('reuse_mix', 'Mixture + RE-USE vs Clean GT'),
    ('reuse_s1', 'Clean s1 + RE-USE vs Clean GT'),
    ('reuse_s1_noisy_40', 'Noisy 40% + RE-USE vs Clean GT'),
]

print(f"\n\n{'='*80}")
print(f"  COMPARISONS AGAINST CLEAN GT")
print(f"{'='*80}")

for cond_key, label in comparisons:
    cond = results.get(cond_key, {})
    if not cond:
        print(f"\n  {label}: NOT AVAILABLE")
        continue

    common = set(gt.keys()) & set(cond.keys())
    matched = sum(1 for f in common if set(gt[f]) == set(cond[f]))
    total = len(common)

    added_counter = Counter()
    removed_counter = Counter()
    for f in common:
        added = set(cond[f]) - set(gt[f])
        removed = set(gt[f]) - set(cond[f])
        added_counter.update(added)
        removed_counter.update(removed)

    print(f"\n  {label}:")
    print(f"    Matched clean GT: {matched}/{total} ({matched/total*100:.1f}%)")
    print(f"    Changed:          {total-matched}/{total} ({(total-matched)/total*100:.1f}%)")
    print(f"    Top artifacts added:")
    for lbl, cnt in added_counter.most_common(5):
        print(f"      + {lbl:<15} {cnt:>5} ({cnt/total*100:.1f}%)")
    print(f"    Top traits lost:")
    for lbl, cnt in removed_counter.most_common(5):
        print(f"      - {lbl:<15} {cnt:>5} ({cnt/total*100:.1f}%)")

print(f"\n{'='*80}")
print(f"  Full results at: {RESULTS_PATH}")
print(f"{'='*80}")
